In [14]:

# ============================================================
# NOTEBOOK 09 — DATASET AUGMENTATION & CONSOLIDATION
# NDCLD15 | Preserve real images + available synthetic samples
# ============================================================

from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import hashlib
import json
import random
import shutil
import zipfile
import time

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
start_time = time.time()

PROJECT = Path("/kaggle/working/nd_iris_project")

REAL_DIR = PROJECT / "notebook02_preprocessing/images_224"
TRAD_ROOT = PROJECT / "notebook05_traditional_augmentation"
GAN_DIR = PROJECT / "notebook06_gan_augmentation/generated_samples"
DIFF_DIR = PROJECT / "notebook07_diffusion_training/generated_samples"

OUT = PROJECT / "notebook09_dataset_augmentation"
DATASET_DIR = OUT / "dataset"
OUT.mkdir(parents=True, exist_ok=True)

# Start clean so reruns do not mix old and new outputs.
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def discover_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTS
    )

# Find likely traditional-augmentation output folders.
# Do not mistake the Notebook 02 source dataset for augmented images.
trad_candidates = discover_images(TRAD_ROOT)
trad_files = [
    p for p in trad_candidates
    if not any(
        token in str(p.relative_to(TRAD_ROOT)).lower()
        for token in [
            "originals/", "source_images/", "real_images/",
            "notebook05_conclusion"
        ]
    )
]

source_groups = {
    "real": discover_images(REAL_DIR),
    "traditional": trad_files,
    "gan": discover_images(GAN_DIR),
    "diffusion": discover_images(DIFF_DIR)
}

print("=" * 65)
print("NOTEBOOK 09 — SOURCE DISCOVERY")
print("=" * 65)
for name, files in source_groups.items():
    print(f"{name:15s}: {len(files)} image files")

if not source_groups["real"]:
    raise FileNotFoundError(
        f"No real images found at {REAL_DIR}. Run Notebook 02 first."
    )

# ------------------------------------------------------------
# 1. Copy images, validate them, and deduplicate by file hash
# ------------------------------------------------------------
manifest_rows = []
seen_hashes = {}
errors = []
duplicate_count = 0
valid_count = 0

for source_name, files in source_groups.items():
    if not files:
        continue

    destination_dir = DATASET_DIR / source_name
    destination_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing {source_name}: {len(files)} files")

    for i, src in enumerate(files):
        try:
            # Validate readability and record dimensions.
            with Image.open(src) as im:
                im.verify()
            with Image.open(src) as im:
                width, height = im.size
                mode = im.mode

            # Hash original bytes to detect exact duplicate files.
            sha256 = hashlib.sha256(src.read_bytes()).hexdigest()

            if sha256 in seen_hashes:
                duplicate_count += 1
                manifest_rows.append({
                    "source": source_name,
                    "original_path": str(src),
                    "dataset_path": seen_hashes[sha256],
                    "sha256": sha256,
                    "width": width,
                    "height": height,
                    "mode": mode,
                    "status": "duplicate_not_copied"
                })
                continue

            # Use a unique, deterministic destination filename.
            dest_name = f"{i:06d}_{sha256[:12]}.png"
            dest = destination_dir / dest_name

            # Normalize dataset copies to grayscale PNG.
            with Image.open(src) as im:
                normalized = im.convert("L")
                normalized.save(dest, format="PNG")

            # Hash normalized copy for future audit.
            copied_hash = hashlib.sha256(dest.read_bytes()).hexdigest()
            seen_hashes[sha256] = str(dest.relative_to(OUT))

            manifest_rows.append({
                "source": source_name,
                "original_path": str(src),
                "dataset_path": str(dest.relative_to(OUT)),
                "sha256": sha256,
                "normalized_sha256": copied_hash,
                "width": width,
                "height": height,
                "mode": mode,
                "status": "copied"
            })
            valid_count += 1

        except Exception as e:
            errors.append({
                "source": source_name,
                "path": str(src),
                "error": str(e)
            })

manifest = pd.DataFrame(manifest_rows)
error_df = pd.DataFrame(errors)

manifest_path = OUT / "dataset_manifest.csv"
manifest.to_csv(manifest_path, index=False)
error_df.to_csv(OUT / "dataset_errors.csv", index=False)

# ------------------------------------------------------------
# 2. Dataset summary and source audit
# ------------------------------------------------------------
copied_manifest = manifest[
    manifest["status"] == "copied"
] if not manifest.empty else pd.DataFrame()

if not copied_manifest.empty:
    source_summary = (
        copied_manifest.groupby("source")
        .agg(
            images=("dataset_path", "count"),
            unique_normalized_files=("normalized_sha256", "nunique")
        )
        .reset_index()
    )
else:
    source_summary = pd.DataFrame(
        columns=["source", "images", "unique_normalized_files"]
    )

source_summary.to_csv(OUT / "dataset_source_summary.csv", index=False)

summary = {
    "notebook": 9,
    "dataset": "NDCLD15",
    "source_discovered_counts": {
        name: len(files) for name, files in source_groups.items()
    },
    "copied_unique_images": int(valid_count),
    "duplicate_source_files_skipped": int(duplicate_count),
    "errors": int(len(error_df)),
    "output_format": "grayscale PNG",
    "class_labels": "not assigned; verified labels unavailable",
    "identity_split": (
        "not performed; no verified identity annotations used here"
    ),
    "elapsed_seconds": round(time.time() - start_time, 2)
}

with open(OUT / "Notebook09_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# ------------------------------------------------------------
# 3. Final Markdown conclusion based on actual run
# ------------------------------------------------------------
lines = [
    "# Notebook 09 — Dataset Augmentation: Conclusion",
    "",
    "## Objective",
    "Consolidate available real and generated NDCLD15 images into a "
    "source-tracked dataset, validate image readability, normalize copied "
    "images to grayscale PNG, and skip exact duplicate source files.",
    "",
    "## Source discovery"
]
for name, files in source_groups.items():
    lines.append(f"- **{name.title()}:** {len(files)} files discovered.")

lines += [
    "",
    "## Processing results",
    f"- Unique images copied: **{valid_count}**",
    f"- Exact duplicate source files skipped: **{duplicate_count}**",
    f"- Processing errors: **{len(error_df)}**",
    "- Output format: grayscale PNG",
    "- Manifest: `dataset_manifest.csv`",
    "- Source summary: `dataset_source_summary.csv`",
    "",
    "## Limitations",
    "Verified contact-lens class labels were not available. The source names "
    "in the manifest identify where each file came from; they are not "
    "contact-lens class labels or verified identity labels. Exact-byte "
    "duplicate detection does not detect visually similar images. Combining "
    "synthetic images into a dataset does not prove they improve recognition.",
    "",
    "## Conclusion",
    "The notebook consolidates the image sources available at runtime and "
    "records their provenance. Review the source summary before using this "
    "dataset for downstream experiments. If a source has zero discovered "
    "files, it was not included.",
    ""
]

conclusion_path = OUT / "Notebook09_Conclusion.md"
conclusion_path.write_text("\n".join(lines), encoding="utf-8")

# ------------------------------------------------------------
# 4. Archive the dataset and all supporting results
# ------------------------------------------------------------
zip_path = PROJECT / "ND_Iris_Notebook09_Results.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file() and p != zip_path:
            zf.write(p, arcname=str(p.relative_to(OUT)))

with zipfile.ZipFile(zip_path, "r") as zf:
    bad_member = zf.testzip()
    archive_count = len(zf.namelist())

# ------------------------------------------------------------
# 5. Display actual results
# ------------------------------------------------------------
print("\n" + "=" * 65)
print("NOTEBOOK 09 — FINAL RESULTS")
print("=" * 65)
print(json.dumps(summary, indent=2))
print("\nSource summary:")
display(source_summary)
print("\nManifest preview:")
display(manifest.head(10))
print("\nConclusion:", conclusion_path)
print("Results ZIP:", zip_path)
print("Archived files:", archive_count)
print("ZIP integrity:", "PASSED" if bad_member is None else f"FAILED: {bad_member}")
print("Output folder:", OUT)


NOTEBOOK 09 — SOURCE DISCOVERY
real           : 6233 image files
traditional    : 12467 image files
gan            : 38 image files
diffusion      : 17 image files

Processing real: 6233 files

Processing traditional: 12467 files

Processing gan: 38 files

Processing diffusion: 17 files

NOTEBOOK 09 — FINAL RESULTS
{
  "notebook": 9,
  "dataset": "NDCLD15",
  "source_discovered_counts": {
    "real": 6233,
    "traditional": 12467,
    "gan": 38,
    "diffusion": 17
  },
  "copied_unique_images": 18740,
  "duplicate_source_files_skipped": 15,
  "errors": 0,
  "output_format": "grayscale PNG",
  "class_labels": "not assigned; verified labels unavailable",
  "identity_split": "not performed; no verified identity annotations used here",
  "elapsed_seconds": 132.98
}

Source summary:


,source,images,unique_normalized_files
0,diffusion,17,17
1,gan,38,38
2,real,6218,6218
3,traditional,12467,12467



Manifest preview:


,source,original_path,dataset_path,sha256,normalized_sha256,width,height,mode,status
0,real,/kaggle/working/nd_iris_project/notebook02_pre...,dataset/real/000000_b84e6c4471ee.png,b84e6c4471ee83b14cb5f9d9629147074a7c286d60bbf9...,b84e6c4471ee83b14cb5f9d9629147074a7c286d60bbf9...,224,224,L,copied
1,real,/kaggle/working/nd_iris_project/notebook02_pre...,dataset/real/000001_54b65a8dd820.png,54b65a8dd820374d4f99045299491b5348f0df1ce118b8...,54b65a8dd820374d4f99045299491b5348f0df1ce118b8...,224,224,L,copied
2,real,/kaggle/working/nd_iris_project/notebook02_pre...,dataset/real/000002_ea127c04a121.png,ea127c04a121e711b7cd15d292f1dc7d61b97183af9fac...,ea127c04a121e711b7cd15d292f1dc7d61b97183af9fac...,224,224,L,copied
3,real,/kaggle/working/nd_iris_project/notebook02_pre...,dataset/real/000003_fc128e55ab7c.png,fc128e55ab7c1ddb4daf791e9e97a240c567b2d1709865...,fc128e55ab7c1ddb4daf791e9e97a240c567b2d1709865...,224,224,L,copied
4,real,/kaggle/working/nd_iris_project/notebook02_pre...,dataset/real/000004_ab2c6505e59e.png,ab2c6505e59e735cf8cb6420914f740475b4d2ebc45609...,ab2c6505e59e735cf8cb6420914f740475b4d2ebc45609...,224,224,L,copied
5,real,/kaggle/working/nd_iris_project/notebook02_pre...,dataset/real/000005_74bf421c086a.png,74bf421c086a19f3f0749e3b2a46eee07c4bd6bff5a3eb...,74bf421c086a19f3f0749e3b2a46eee07c4bd6bff5a3eb...,224,224,L,copied
6,real,/kaggle/working/nd_iris_project/notebook02_pre...,dataset/real/000006_86c3e8186b49.png,86c3e8186b492d32c396fdbd7e784b3eec278ac2281c93...,86c3e8186b492d32c396fdbd7e784b3eec278ac2281c93...,224,224,L,copied
7,real,/kaggle/working/nd_iris_project/notebook02_pre...,dataset/real/000007_d726ce3e3c5f.png,d726ce3e3c5f33dc32d2e920becef999778e9c0e9b0cd2...,d726ce3e3c5f33dc32d2e920becef999778e9c0e9b0cd2...,224,224,L,copied
8,real,/kaggle/working/nd_iris_project/notebook02_pre...,dataset/real/000008_749688e936af.png,749688e936af714585fbed02772fcbc1a05996242d4aed...,749688e936af714585fbed02772fcbc1a05996242d4aed...,224,224,L,copied
9,real,/kaggle/working/nd_iris_project/notebook02_pre...,dataset/real/000009_32f23fd59b30.png,32f23fd59b30c8f512c7f03651b4480ff3188a176ebfc3...,32f23fd59b30c8f512c7f03651b4480ff3188a176ebfc3...,224,224,L,copied



Conclusion: /kaggle/working/nd_iris_project/notebook09_dataset_augmentation/Notebook09_Conclusion.md
Results ZIP: /kaggle/working/nd_iris_project/ND_Iris_Notebook09_Results.zip
Archived files: 18745
ZIP integrity: PASSED
Output folder: /kaggle/working/nd_iris_project/notebook09_dataset_augmentation


# Notebook 09 — Dataset Augmentation: Conclusion

## Objective
Consolidate real NDCLD15 images and available traditional, GAN, and diffusion outputs into a source-tracked dataset.

## Source Discovery
- Real images discovered: 6,233
- Traditional augmentation images discovered: 12,467
- GAN-generated images discovered: 38
- Diffusion-generated images discovered: 17

## Processing Results
- Unique images copied: **18,740**
- Exact duplicate source files skipped: **15**
- Processing errors: **0**
- Output format: grayscale PNG
- ZIP integrity: **Passed**

## Final Dataset Composition
- Real: 6,218 images
- Traditional augmentation: 12,467 images
- GAN: 38 images
- Diffusion: 17 images
- **Total: 18,740 images**

## Generated Artifacts
The notebook generated a dataset manifest, source summary, error report, JSON summary, and conclusion report. The dataset and supporting artifacts were archived in `ND_Iris_Notebook09_Results.zip`.

## Limitations
Verified contact-lens class labels and identity annotations were not available. Source categories indicate image provenance, not contact-lens classes or verified identities. Exact-byte duplicate detection does not identify visually similar images. Dataset consolidation alone does not establish improved model performance.

## Conclusion
The dataset consolidation pipeline completed successfully, producing 18,740 unique copied images with zero processing errors. The resulting dataset is ready for carefully controlled exploratory experiments, but verified labels and appropriate data splitting are required before making claims about contact-lens classification or biometric recognition.